In [5]:
import pandas as pd
import numpy as np
import os


In [3]:
hnei_dataset = pd.read_csv("Battery_RUL.csv")

In [4]:
meta_path = 'cleaned_dataset/metadata.csv'
data_dir = 'cleaned_dataset/data/'
metadata = pd.read_csv(meta_path)

In [6]:
extracted_features = []

# 2. Process each battery
for battery_id, bat_group in metadata.groupby('battery_id'):
    cycle_index = 1
    
    charges = bat_group[bat_group['type'] == 'charge'].reset_index()
    discharges = bat_group[bat_group['type'] == 'discharge'].reset_index()
    
    max_cycles = min(len(charges), len(discharges))
    
    for i in range(max_cycles):
        charge_meta = charges.iloc[i]
        discharge_meta = discharges.iloc[i]
        
        # Load raw time-series CSVs
        charge_df = pd.read_csv(os.path.join(data_dir, charge_meta['filename']))
        discharge_df = pd.read_csv(os.path.join(data_dir, discharge_meta['filename']))
        
        # --- Feature Extraction ---
        
        # Min. Voltage Charg. (V)
        min_v_charge = charge_df['Voltage_measured'].min()
        
        # Charging time (s)
        charge_time = charge_df['Time'].max()
        
        # Time constant current (s) (NASA charges at 1.5A, tracking > 1.4A)
        cc_phase = charge_df[charge_df['Current_measured'] > 1.4]
        time_cc = cc_phase['Time'].max() - cc_phase['Time'].min() if not cc_phase.empty else 0
        
        # Time at 4.15V (s)
        high_v_phase = charge_df[charge_df['Voltage_measured'] >= 4.15]
        time_415v = high_v_phase['Time'].max() - high_v_phase['Time'].min() if not high_v_phase.empty else 0

        # Discharge Time (s)
        discharge_time = discharge_df['Time'].max()
        
        # Max. Voltage Dischar. (V)
        max_v_discharge = discharge_df['Voltage_measured'].max()
        
        # Decrement 3.6-3.4V (s)
        try:
            time_36 = discharge_df[discharge_df['Voltage_measured'] <= 3.6]['Time'].iloc[0]
            time_34 = discharge_df[discharge_df['Voltage_measured'] <= 3.4]['Time'].iloc[0]
            decrement_time = time_34 - time_36
        except IndexError:
            decrement_time = np.nan 

        # Append with EXACT HNEI headers
        extracted_features.append({
            'Battery_ID': battery_id,
            'Cycle_Index': cycle_index,
            'Discharge Time (s)': discharge_time,
            'Decrement 3.6-3.4V (s)': decrement_time,
            'Max. Voltage Dischar. (V)': max_v_discharge,
            'Min. Voltage Charg. (V)': min_v_charge,
            'Time at 4.15V (s)': time_415v,
            'Time constant current (s)': time_cc,
            'Charging time (s)': charge_time
        })
        
        cycle_index += 1

In [7]:
# 3. Create DataFrame and clean missing values
nasa_df = pd.DataFrame(extracted_features)
nasa_df.dropna(inplace=True)

# 4. Calculate RUL
# For each battery, find its max cycle index, then subtract the current cycle index
nasa_df['RUL'] = nasa_df.groupby('Battery_ID')['Cycle_Index'].transform('max') - nasa_df['Cycle_Index']

# (Optional) Drop the Battery_ID column if you strictly want to match the HNEI columns
nasa_df = nasa_df.drop(columns=['Battery_ID'])

In [8]:
print(nasa_df.head())

   Cycle_Index  Discharge Time (s)  Decrement 3.6-3.4V (s)  \
0            1            3690.234                1488.734   
1            2            3672.344                1472.266   
2            3            3651.641                1471.641   
3            4            3631.563                1472.532   
4            5            3629.172                1471.328   

   Max. Voltage Dischar. (V)  Min. Voltage Charg. (V)  Time at 4.15V (s)  \
0                   4.191492                 3.479394           7249.672   
1                   4.189773                 3.001951           7609.047   
2                   4.188187                 3.035879           7568.453   
3                   4.188461                 3.066145           7491.562   
4                   4.188299                 3.063766           7589.563   

   Time constant current (s)  Charging time (s)  RUL  
0                    760.250           7597.875  167  
1                   3367.391          10516.000  166  
2    

In [9]:
# save to csv called "nasa_battery_cycles.csv"
nasa_df.to_csv("nasa_battery_cycles.csv", index=False)

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load the data
hnei_df = pd.read_csv('Battery_RUL.csv')
nasa_df = pd.read_csv('nasa_battery_cycles.csv')

# 2. Clean HNEI Outliers 
# Dropping impossible values (like negative time or phases over 100,000 seconds)
time_cols = ['Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Time at 4.15V (s)', 
             'Time constant current (s)', 'Charging time (s)']

for col in time_cols:
    hnei_df = hnei_df[(hnei_df[col] >= 0) & (hnei_df[col] < 100000)]

# 3. Add Dataset Identifier (The Origin Flag)
hnei_df['Is_NASA'] = 0
nasa_df['Is_NASA'] = 1

# 4. Scale Features INDIVIDUALLY per dataset
feature_cols = ['Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Max. Voltage Dischar. (V)', 
                'Min. Voltage Charg. (V)', 'Time at 4.15V (s)', 'Time constant current (s)', 'Charging time (s)']

scaler_hnei = StandardScaler()
hnei_df_scaled = hnei_df.copy()
hnei_df_scaled[feature_cols] = scaler_hnei.fit_transform(hnei_df[feature_cols])

scaler_nasa = StandardScaler()
nasa_df_scaled = nasa_df.copy()
nasa_df_scaled[feature_cols] = scaler_nasa.fit_transform(nasa_df[feature_cols])

# 5. Combine the Datasets
combined_df = pd.concat([hnei_df_scaled, nasa_df_scaled], ignore_index=True)

# Save the prepared dataset!
combined_df.to_csv('combined_scaled_battery_data.csv', index=False)